<a href="https://colab.research.google.com/github/PauloRadatz/opendss-python-examples/blob/main/presentations/ieee_et_pes_pels_joint_chapter_workshop/02_load_level_sweep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 2 — Why Python automation matters: load level sweep

In Notebook 1 we replicated a run file in Python. That alone does not show why Python is worth the effort. The real value shows up when we want to repeat the same study under many conditions — that is when typing commands by hand stops being practical.

In this notebook we run the **same feeder under many load levels** (peak, off-peak, and several points in between) and collect the results in a single table and plot. Doing this by hand in OpenDSS would mean editing `loadmult`, clicking solve, copying numbers, twelve times in a row. With a Python loop it is one cell.

## Setup

In [ ]:
!pip install py-dss-interface
!git clone https://github.com/PauloRadatz/opendss-python-examples.git
%cd opendss-python-examples

In [ ]:
import pathlib
import pandas as pd
import matplotlib.pyplot as plt
import py_dss_interface

start_dir = pathlib.Path.cwd().resolve()
repo_root = next(
    parent for parent in [start_dir, *start_dir.parents]
    if (parent / "feeder_models").exists()
)
dss_file = repo_root / "feeder_models" / "EPRITestCircuits" / "ckt5" / "Master_ckt5.dss"

dss = py_dss_interface.DSS()
print(f"Using feeder file: {dss_file}")

## Define the load levels we want to study

We sweep `loadmult` from a light off-peak condition (0.2) up to peak load (1.0) in steps of 0.1. Nine power flows in total — easy in Python, painful by hand.

In [ ]:
load_multipliers = [round(0.2 + 0.1 * i, 2) for i in range(9)]
print(load_multipliers)

## Loop: compile, set load, solve, read results

For each load level we recompile the feeder, set `loadmult`, solve a snapshot, and capture three quantities:

- minimum voltage on the feeder (pu)
- maximum voltage on the feeder (pu)
- total active losses (kW)
- active power at the feederhead (kW)

Recompiling each time guarantees a clean state — that is the safest pattern when sweeping conditions.

In [ ]:
results = []

for load_mult in load_multipliers:
    dss.text(f"compile [{dss_file}]")
    dss.text(f"set loadmult={load_mult}")
    dss.text("solve")

    voltages_pu = dss.circuit.buses_vmag_pu
    p_kw, _ = dss.circuit.total_power
    p_loss_w, _ = dss.circuit.losses

    results.append({
        "Load multiplier": load_mult,
        "Feederhead P (kW)": -p_kw,
        "Total losses (kW)": p_loss_w / 1000.0,
        "Min voltage (pu)": min(voltages_pu),
        "Max voltage (pu)": max(voltages_pu),
    })

results_df = pd.DataFrame(results)
results_df

## Plot voltage range vs. load level

Two horizontal reference lines mark the typical 0.95 / 1.05 pu service band so the operating window is easy to read.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

ax.plot(results_df["Load multiplier"], results_df["Min voltage (pu)"], marker="o", label="Min voltage")
ax.plot(results_df["Load multiplier"], results_df["Max voltage (pu)"], marker="o", label="Max voltage")
ax.axhline(0.95, color="red", linestyle="--", linewidth=1, label="0.95 pu")
ax.axhline(1.05, color="red", linestyle="--", linewidth=1, label="1.05 pu")
ax.set_xlabel("Load multiplier")
ax.set_ylabel("Voltage (pu)")
ax.set_title("Min / Max voltage across the feeder vs. load level")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## Plot losses and feederhead power vs. load level

Losses scale roughly with the square of the current, so they grow faster than the load itself. The plot makes that pattern visible.

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 4.5))

color1 = "tab:blue"
ax1.set_xlabel("Load multiplier")
ax1.set_ylabel("Feederhead P (kW)", color=color1)
ax1.plot(results_df["Load multiplier"], results_df["Feederhead P (kW)"], marker="o", color=color1, label="Feederhead P")
ax1.tick_params(axis="y", labelcolor=color1)
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
color2 = "tab:red"
ax2.set_ylabel("Total losses (kW)", color=color2)
ax2.plot(results_df["Load multiplier"], results_df["Total losses (kW)"], marker="s", color=color2, label="Total losses")
ax2.tick_params(axis="y", labelcolor=color2)

plt.title("Feederhead power and losses vs. load level")
plt.tight_layout()
plt.show()

## Key takeaways

- The same `compile / set loadmult / solve / read results` pattern from Notebook 1 is now wrapped in a `for` loop. Nothing more.
- Once the workflow lives in Python, sweeping one parameter (load level) is a few lines. Sweeping two (load and PV penetration, for example) is the same loop with one more dimension.
- A pandas DataFrame and a matplotlib plot are usually the right way to *see* the result. Pick the abstractions that match the question.

This is the practical reason Python pairs well with OpenDSS: studies that would be tedious by hand become straightforward, and the analysis is reproducible the next time we open the notebook.

## Additional learning resources

If you would like to continue learning OpenDSS and Python for power-system studies, you can find more educational materials and courses here:

- [pauloradatz.me](https://www.pauloradatz.me)
- [OpenDSS courses](https://www.pauloradatz.me/opendss-courses)

## Contact

For questions or follow-up about these materials:

- Paulo Radatz
- Email: [paulo.radatz@gmail.com](mailto:paulo.radatz@gmail.com)
- LinkedIn: [linkedin.com/in/pauloradatz](https://www.linkedin.com/in/pauloradatz/)
- Website: [pauloradatz.me](https://www.pauloradatz.me/)